In [2]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from bs4 import BeautifulSoup as bs
import requests
from tqdm import tqdm
from time import sleep
import json
import sys

In [23]:
matches_df = pd.read_csv("top_player_matches_7-24-2022.csv").sort_values('Match',ascending=False,ignore_index=True)

In [24]:
matches = matches_df['Match'].values[5000:]

In [3]:
df = pd.read_csv('first_5000_compact_match_data.csv')

In [26]:
def reduce_memory(df):
    """
    Modifies DataFrame passed to reduce memory usage
    """
    init_mem_usage = round(df.memory_usage().sum() / (1024**2),2)
    for col in df.columns:
        dtype = df[col].dtype
        altered = 'object'
        if dtype != 'object':
            if dtype == 'float64':
                f32 = df[col].astype('float32')
                diff = np.abs(df[col]-f32).sum()
                if diff < 0.1:
                    df[col] = df[col].astype('float32')
                    
                f16 = df[col].astype('float16')
                diff = np.abs(df[col]-f16).sum()
                if diff < 0.01:
                    df[col] = df[col].astype('float16')
            else:
                mn = df[col].min()
                mx = df[col].max()
                if mn >= 0:
                    if mx < np.iinfo(np.uint8).max:
                        df[col] = df[col].astype('uint8')
                    elif mx < np.iinfo(np.uint16).max:
                        df[col] = df[col].astype('uint16')
                    elif mx < np.iinfo(np.uint32).max:
                        df[col] = df[col].astype('uint32')
                    else:
                        df[col] = df[col].astype('uint64')
                else:
                    if mn > np.iinfo(np.int8).min and mx < np.iinfo(np.int8).max:
                        df[col] = df[col].astype('int8')
                    elif mn > np.iinfo(np.int16).min and mx < np.iinfo(np.int16).max:
                        df[col] = df[col].astype('int16')
                    elif mn > np.iinfo(np.int32).min and mx < np.iinfo(np.int32).max:
                        df[col] = df[col].astype('int32')
            altered = df[col].dtype
        
        print("Column {} \t{} -> {}".format(col,dtype,altered))
    
    final_mem_usage = round(df.memory_usage().sum() / (1024**2),2)
    print(f"\nInitial Mem Usage: {init_mem_usage} MB")
    print(f"Final Mem Usage: {final_mem_usage} MB")
    print("Memory Change PCT: {}%".format(round(100-100*final_mem_usage/init_mem_usage,2)))

In [27]:
#reduce_memory(df)

https://api.opendota.com/api/matches/{match_id}

matches found up to 1000

api key: c4b80265-46ad-4728-b0f4-ae0c1a175bef

In [28]:
len(matches)

5348

In [29]:
def get_match_data(matches):
    match_data = []
    errors = []
    for match in tqdm(matches,total=len(matches)):
        try:
            request = requests.request('GET',f"https://api.opendota.com/api/matches/{match}?api_key=c4b80265-46ad-4728-b0f4-ae0c1a175bef")
            data = request.json()
            if(data['players']):
                match_data.append(data)
        except:
            errors.append(match)
    
    df = pd.DataFrame(match_data)
    
    return df, errors
    

In [30]:
df, errors = get_match_data(matches)

100%|██████████████████████████████████████████████████████████████████████████████| 5348/5348 [59:03<00:00,  1.51it/s]


In [31]:
len(errors)

0

In [9]:
#df.to_csv('first_5000_matches_full.csv',index=False)

In [32]:
FANTASY_METRICS = ['match_id',
        'account_id',
        'assists',
        'camps_stacked',
        'deaths',
        'denies',
        'firstblood_claimed',
        'gold_per_min',
        'kills',
        'last_hits',
        'obs_placed',
        'roshans_killed',
        'rune_pickups',
        'stuns',
        'teamfight_participation',
        'towers_killed',
        'start_time',
        'radiant_win',
        'isRadiant',
        'win',
        'lose']

In [5]:
MATCH_DROP = ['barracks_status_dire',
              'barracks_status_radiant',
              'dire_score',
              'radiant_score',
              'chat',
              'cluster',
              'cosmetics',
              'draft_timings',
              'first_blood_time',
              'human_players',
              'league',
              'match_seq_num',
              'negative_votes',
              'objectives',
              'picks_bans',
              'positive_votes',
              'radiant_gold_adv',
              'radiant_xp_adv',
              'skill',
              'start_time',
              'teamfights',
              'tower_status_dire',
              'tower_status_radiant',
              'radiant_team',
              'dire_team',
              'version',
              'players',
              'replay_salt',
              'all_word_counts',
              'my_word_counts',
              'throw',
              'loss',
              'replay_url',
              'comeback',
              'stomp']

In [11]:
#full_match_df = pd.DataFrame(matches)
#full_match_df.to_csv('full_match_data.csv',index=False)
#full_match_df = pd.read_csv('full_match_data.csv')

In [37]:
end_5000_players = df['players'].values

In [7]:
#compact_match_df = df.drop(MATCH_DROP,axis=1)
compact_match_df = df.drop(['radiant_score','dire_score','players'],axis=1)
compact_match_df.to_csv('first_5000_compact_match_data.csv',index=False)

In [67]:
compact_match_df.head()

,match_id,cluster,dire_score,dire_team_id,engine,game_mode,leagueid,lobby_type,radiant_score,radiant_team_id,radiant_win,series_id,series_type,radiant_team,dire_team,players,patch,region
0,6649226879,121,8,8261882,1,2,14281,1,24,39,True,682742,1,"{'team_id': 39, 'name': 'Evil Geniuses', 'tag'...","{'team_id': 8261882, 'name': 'felt', 'tag': 'f...","[{'match_id': 6649226879, 'player_slot': 0, 'a...",50,2.0
1,6649166938,121,28,39,1,2,14281,1,10,8261882,False,682742,1,"{'team_id': 8261882, 'name': 'felt', 'tag': 'f...","{'team_id': 39, 'name': 'Evil Geniuses', 'tag'...","[{'match_id': 6649166938, 'player_slot': 0, 'a...",50,2.0
2,6649082276,273,14,8599101,1,2,14279,1,31,1838315,True,682693,1,"{'team_id': 1838315, 'name': 'Team Secret', 't...","{'team_id': 8599101, 'name': 'Gaimin Gladiator...","[{'match_id': 6649082276, 'player_slot': 0, 'a...",50,3.0
3,6648984612,273,35,1838315,1,2,14279,1,38,8599101,False,682693,1,"{'team_id': 8599101, 'name': 'Gaimin Gladiator...","{'team_id': 1838315, 'name': 'Team Secret', 't...","[{'match_id': 6648984612, 'player_slot': 0, 'a...",50,3.0
4,6648906883,274,33,8599101,1,2,14279,1,8,1838315,False,682693,1,"{'team_id': 1838315, 'name': 'Team Secret', 't...","{'team_id': 8599101, 'name': 'Gaimin Gladiator...","[{'match_id': 6648906883, 'player_slot': 0, 'a...",50,3.0


In [65]:
compact_match_df[compact_match_df['series_type'] == 2].groupby('series_id').count()

,match_id,cluster,dire_score,dire_team_id,engine,game_mode,leagueid,lobby_type,radiant_score,radiant_team_id,radiant_win,series_type,radiant_team,dire_team,players,patch,region
series_id,,,,,,,,,,,,,,,,,
634916,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3
642144,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,3
642978,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
644221,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
644482,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5
646747,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3
648409,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5
649122,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3
655167,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,2


In [9]:
list(full_match_df.columns)

['match_id',
 'barracks_status_dire',
 'barracks_status_radiant',
 'chat',
 'cluster',
 'cosmetics',
 'dire_score',
 'dire_team_id',
 'draft_timings',
 'duration',
 'engine',
 'first_blood_time',
 'game_mode',
 'human_players',
 'leagueid',
 'lobby_type',
 'match_seq_num',
 'negative_votes',
 'objectives',
 'picks_bans',
 'positive_votes',
 'radiant_gold_adv',
 'radiant_score',
 'radiant_team_id',
 'radiant_win',
 'radiant_xp_adv',
 'skill',
 'start_time',
 'teamfights',
 'tower_status_dire',
 'tower_status_radiant',
 'version',
 'replay_salt',
 'series_id',
 'series_type',
 'league',
 'radiant_team',
 'dire_team',
 'players',
 'patch',
 'region',
 'all_word_counts',
 'my_word_counts',
 'throw',
 'loss',
 'replay_url',
 'comeback',
 'stomp']

In [51]:
def make_player_df(players):
    all_matches = []
    for player in tqdm(players,total=len(players)):
        all_matches.append(pd.DataFrame(eval(str(player)))[FANTASY_METRICS])

    player_match_df = pd.concat(all_matches,ignore_index=True)
    
    return player_match_df

In [52]:
sec_player_match_df = make_player_df(end_5000_players)

100%|██████████████████████████████████████████████████████████████████████████████| 5348/5348 [11:11<00:00,  7.96it/s]


In [53]:
sec_player_match_df

,match_id,account_id,assists,camps_stacked,deaths,denies,firstblood_claimed,gold_per_min,kills,last_hits,...,roshans_killed,rune_pickups,stuns,teamfight_participation,towers_killed,start_time,radiant_win,isRadiant,win,lose
0,6185613522,135495981,11,1.0,17,4,0.0,375,1,178,...,0.0,2.0,123.179540,0.571429,0.0,1631961264,False,True,0,1
1,6185613522,173378701,7,6.0,12,7,0.0,410,7,198,...,0.0,10.0,52.534863,0.619048,0.0,1631961264,False,True,0,1
2,6185613522,323792491,12,1.0,16,0,1.0,222,2,26,...,0.0,4.0,32.974194,0.619048,0.0,1631961264,False,True,0,1
3,6185613522,156029808,8,4.0,10,4,0.0,332,8,92,...,0.0,3.0,82.676620,0.714286,0.0,1631961264,False,True,0,1
4,6185613522,337575662,8,2.0,8,20,0.0,490,3,278,...,0.0,1.0,38.150080,0.523810,0.0,1631961264,False,True,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53475,644342976,38271450,10,NaN,0,0,NaN,408,8,19,...,NaN,NaN,NaN,NaN,NaN,1399295805,False,False,1,0
53476,644342976,86827378,6,NaN,0,3,NaN,331,1,28,...,NaN,NaN,NaN,NaN,NaN,1399295805,False,False,1,0
53477,644342976,66598119,11,NaN,2,1,NaN,410,4,54,...,NaN,NaN,NaN,NaN,NaN,1399295805,False,False,1,0
53478,644342976,86874930,10,NaN,0,22,NaN,610,10,92,...,NaN,NaN,NaN,NaN,NaN,1399295805,False,False,1,0


In [54]:
player_match_df = pd.concat([player_match_df,sec_player_match_df],ignore_index=True)

In [55]:
player_match_df.to_csv('player_data.csv',index=False)

In [56]:
player_match_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103480 entries, 0 to 103479
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   match_id                 103480 non-null  int64  
 1   account_id               103480 non-null  int64  
 2   assists                  103480 non-null  int64  
 3   camps_stacked            99774 non-null   object 
 4   deaths                   103480 non-null  int64  
 5   denies                   103480 non-null  int64  
 6   firstblood_claimed       98801 non-null   object 
 7   gold_per_min             103480 non-null  int64  
 8   kills                    103480 non-null  int64  
 9   last_hits                103480 non-null  int64  
 10  obs_placed               99774 non-null   object 
 11  roshans_killed           99161 non-null   object 
 12  rune_pickups             99774 non-null   object 
 13  stuns                    100258 non-null  float64
 14  team

In [57]:
player_match_df.groupby('account_id').count()

,match_id,assists,camps_stacked,deaths,denies,firstblood_claimed,gold_per_min,kills,last_hits,obs_placed,roshans_killed,rune_pickups,stuns,teamfight_participation,towers_killed,start_time,radiant_win,isRadiant,win,lose
account_id,,,,,,,,,,,,,,,,,,,,
88470,159,159,158,159,159,158,159,159,159,158,158,158,158,158,158,159,159,159,159,159
400183,10,10,7,10,10,7,10,10,10,7,7,7,8,7,7,10,10,10,10,10
522916,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
1296625,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10
2503633,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1216883747,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
1235513697,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4
1261414529,57,57,57,57,57,57,57,57,57,57,57,57,57,57,57,57,57,57,57,57
